# 🐑 Sheep Activity Classifier v3

### Cambios respecto a v2 (por qué val F1=1.0 pero test F1=0.55)

**Problema raíz:** ViT-Base tiene 86M parámetros para solo 100 videos → memoriza el conjunto de validación (que era de ~16 videos). El modelo no aprendía a generalizar.

**Soluciones aplicadas:**
1. **Backbone más pequeño**: `EfficientNet-B0` (5.3M params) en lugar de ViT-Base (86M). Mejor ratio parámetros/datos para datasets pequeños.
2. **Pseudo-labeling**: entrenar con train → predecir test con alta confianza → re-entrenar con train + pseudo-labels del test. Esto aprovecha los 169 videos de test sin etiquetas.
3. **Freeze total del backbone en ronda 1**, solo entrena el clasificador. En ronda 2 se descongela parcialmente (fine-tuning progresivo).
4. **Stochastic Depth** (drop path) como regularización adicional.
5. `n_frames` reducido a **8** (suficiente para capturar la actividad, reduce overfitting por frame).
6. K-Fold mantenido pero con **early stopping más agresivo** (patience=7).

### Por qué NO fusionar train+test para K-Fold
No tenemos etiquetas del test, así que no podemos usarlo directamente en K-Fold supervisado. El **pseudo-labeling** es la alternativa correcta.


## 0 · Instalación

In [ ]:
!pip install -q timm>=0.9.0 ultralytics>=8.0.0 einops

## 1 · GPU, imports y config

In [ ]:
import os, sys, math, random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import timm
from PIL import Image
import torchvision.transforms.functional as TF
import torchvision.transforms as T

assert torch.cuda.is_available(), "⚠️ Activa la GPU en Settings → Accelerator."
DEVICE = torch.device("cuda")
print(f"✅ GPU : {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"   PyTorch {torch.__version__} · CUDA {torch.version.cuda}")

In [ ]:
# ── Rutas ──────────────────────────────────────────────────────────────
DATASET_NAME = "sheep-activity"   # <-- nombre dataset de videos
SCRIPTS_NAME = "sheep-scripts"    # <-- nombre dataset de .py

DATA_DIR        = Path("/kaggle/input") / DATASET_NAME
TRAIN_VIDEO_DIR = DATA_DIR / "train"
TEST_VIDEO_DIR  = Path("/kaggle/working") / "test"   # descomprimido de gdown
LABEL_CSV       = DATA_DIR / "train.csv"

WORKING_DIR    = Path("/kaggle/working")
PROCESSED_DIR  = WORKING_DIR / "processed"
CHECKPOINT_DIR = WORKING_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(Path("/kaggle/input") / SCRIPTS_NAME))

# ── Config ─────────────────────────────────────────────────────────────
CFG = {
    # Preprocesamiento
    "n_frames"         : 8,           # ↓ de 16 → menos overfitting por frame
    "video_ext"        : ".mov",
    # Modelo
    "backbone"         : "efficientnet_b0",  # 5.3M params vs 86M del ViT-B
    "num_classes"      : 5,
    "dropout"          : 0.5,
    "unfreeze_blocks"  : 0,           # Fase 1: backbone 100% congelado
    # K-Fold
    "n_folds"          : 5,
    # Entrenamiento fase 1 (solo clasificador)
    "epochs_phase1"    : 30,
    "lr_phase1"        : 3e-4,
    # Entrenamiento fase 2 (fine-tuning parcial)
    "epochs_phase2"    : 20,
    "lr_phase2"        : 5e-5,
    "unfreeze_phase2"  : 3,           # Descongelar últimos 3 bloques en fase 2
    # Pseudo-labeling
    "pseudo_confidence": 0.85,        # Solo usar predicciones con prob >= 0.85
    "pseudo_epochs"    : 20,
    "pseudo_lr"        : 3e-5,
    # Común
    "batch_size"       : 16,
    "weight_decay"     : 0.01,
    "warmup_epochs"    : 3,
    "patience"         : 7,           # más agresivo que v2
    "mixup_prob"       : 0.4,
    "mixup_alpha"      : 0.3,
    "max_grad_norm"    : 1.0,
    "label_smoothing"  : 0.1,
    "num_workers"      : 2,
    "seed"             : 42,
    "use_tta"          : True,
    "tta_augments"     : 6,
}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CFG["seed"])
print("Config cargada ✅")

## 2 · Modelo EfficientNet (reemplaza ViT-Base)

In [ ]:
# ── Temporal Attention Pool ────────────────────────────────────────────
class TemporalAttentionPool(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 4), nn.Tanh(),
            nn.Linear(embed_dim // 4, 1),
        )
    def forward(self, x):  # x: (B, N, D)
        w = F.softmax(self.attention(x), dim=1)  # (B, N, 1)
        return (w * x).sum(dim=1)                # (B, D)


# ── Clasificador principal ─────────────────────────────────────────────
class SheepClassifier(nn.Module):
    """
    EfficientNet-B0 como backbone (5.3M params, mucho más adecuado para 100 videos)
    + Temporal Attention Pool sobre los N frames
    + MLP clasificador con Dropout fuerte
    """
    def __init__(self, num_classes=5, n_frames=8, backbone_name="efficientnet_b0",
                 dropout=0.5, unfreeze_last_n=0):
        super().__init__()
        self.n_frames = n_frames

        self.backbone = timm.create_model(
            backbone_name, pretrained=True, num_classes=0, global_pool="avg"
        )
        embed_dim = self.backbone.num_features  # 1280 para B0

        self._set_frozen(unfreeze_last_n)
        self.temporal_pool = TemporalAttentionPool(embed_dim)
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout * 0.6),
            nn.Linear(256, num_classes),
        )
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def _set_frozen(self, unfreeze_last_n: int):
        """Congela todo el backbone excepto los últimos N bloques."""
        for p in self.backbone.parameters():
            p.requires_grad = False

        if unfreeze_last_n > 0:
            # EfficientNet tiene bloques en backbone.blocks
            blocks = list(self.backbone.blocks)
            for block in blocks[-unfreeze_last_n:]:
                for p in block.parameters():
                    p.requires_grad = True
            # Descongelar también conv_head y bn2
            for name in ["conv_head", "bn2"]:
                layer = getattr(self.backbone, name, None)
                if layer:
                    for p in layer.parameters():
                        p.requires_grad = True

        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Params totales: {total/1e6:.2f}M | Entrenables: {trainable/1e6:.2f}M ({100*trainable/total:.1f}%)")

    def forward(self, clip):  # clip: (B, N, C, H, W)
        B, N, C, H, W = clip.shape
        frames = clip.view(B * N, C, H, W)
        emb    = self.backbone(frames).view(B, N, -1)  # (B, N, D)
        pooled = self.temporal_pool(emb)               # (B, D)
        return self.classifier(pooled)                 # (B, num_classes)


# ── Label Smoothing con soporte para targets suaves (Mixup) ───────────
class SmoothedCE(nn.Module):
    def __init__(self, smoothing=0.1, num_classes=5):
        super().__init__()
        self.smoothing = smoothing
        self.num_classes = num_classes

    def forward(self, logits, targets):
        log_p = F.log_softmax(logits, dim=-1)
        if targets.dim() == 1:
            oh = torch.zeros_like(log_p).fill_(self.smoothing / (self.num_classes - 1))
            oh.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        else:
            oh = targets * (1 - self.smoothing) + self.smoothing / self.num_classes
        return -(oh * log_p).sum(dim=-1).mean()


def build_model(unfreeze=0):
    m = SheepClassifier(
        num_classes=CFG["num_classes"],
        n_frames=CFG["n_frames"],
        backbone_name=CFG["backbone"],
        dropout=CFG["dropout"],
        unfreeze_last_n=unfreeze,
    ).to(DEVICE)
    return m

print("Arquitectura definida ✅")
# Test rápido de shapes
_m = build_model(0)
_x = torch.randn(2, CFG["n_frames"], 3, 224, 224).to(DEVICE)
with torch.no_grad():
    _out = _m(_x)
print(f"  Output shape: {_out.shape}  (esperado: [2, 5])")
del _m, _x, _out
torch.cuda.empty_cache()

## 3 · Dataset con augmentation mejorada

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 224


class TemporalConsistentTransform:
    """
    Misma transformación espacial para todos los frames del clip.
    Augmentation más fuerte que v2 para compensar dataset pequeño.
    """
    def __init__(self, is_train=True):
        self.is_train = is_train
        self.to_tensor = T.ToTensor()
        self.normalize = T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        self.erase = T.RandomErasing(p=0.35, scale=(0.02, 0.2))

    def __call__(self, frames):
        if self.is_train:
            # Parámetros aleatorios compartidos
            i, j, h, w = T.RandomResizedCrop.get_params(
                frames[0], scale=(0.65, 1.0), ratio=(0.8, 1.2)
            )
            flip_h  = random.random() < 0.5
            flip_v  = random.random() < 0.15
            angle   = random.uniform(-25, 25)
            bright  = random.uniform(0.5, 1.5)
            contrast= random.uniform(0.5, 1.5)
            sat     = random.uniform(0.6, 1.4)
            hue     = random.uniform(-0.15, 0.15)
            do_gray = random.random() < 0.08
            blur_s  = random.uniform(0.1, 2.5) if random.random() < 0.4 else None

        out = []
        for f in frames:
            if self.is_train:
                f = TF.resized_crop(f, i, j, h, w, (IMG_SIZE, IMG_SIZE))
                if flip_h: f = TF.hflip(f)
                if flip_v: f = TF.vflip(f)
                f = TF.rotate(f, angle)
                f = TF.adjust_brightness(f, bright)
                f = TF.adjust_contrast(f, contrast)
                f = TF.adjust_saturation(f, sat)
                f = TF.adjust_hue(f, hue)
                if do_gray: f = TF.rgb_to_grayscale(f, num_output_channels=3)
                if blur_s:  f = TF.gaussian_blur(f, kernel_size=5, sigma=blur_s)
            else:
                f = f.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
            t = self.normalize(self.to_tensor(f))
            if self.is_train:
                t = self.erase(t)
            out.append(t)
        return torch.stack(out)  # (N, C, H, W)


class SheepDataset(Dataset):
    """
    processed_dir : carpeta con subcarpetas por video_id
    labels_df     : DataFrame con [Id, Target], o None para test
    pseudo_df     : DataFrame adicional con pseudo-labels [Id, Target]
    """
    def __init__(self, processed_dir, labels_df=None, pseudo_df=None,
                 n_frames=8, is_train=True):
        self.root     = Path(processed_dir)
        self.n_frames = n_frames
        self.transform = TemporalConsistentTransform(is_train)

        if labels_df is not None:
            df = labels_df.copy()
            if pseudo_df is not None:
                df = pd.concat([df, pseudo_df], ignore_index=True)
            self.samples = [
                (str(row["Id"]), int(row["Target"]))
                for _, row in df.iterrows()
                if (self.root / str(row["Id"])).exists()
            ]
        else:
            self.samples = [
                (d.name, -1)
                for d in sorted(self.root.iterdir()) if d.is_dir()
            ]
        print(f"  Dataset: {len(self.samples)} muestras ({'train' if is_train else 'val/test'})")

    def _load_frames(self, video_id):
        vid_dir = self.root / video_id
        files   = sorted(vid_dir.glob("frame_*.png"))
        frames  = [Image.open(f).convert("RGB") for f in files]
        while len(frames) < self.n_frames:
            frames.append(frames[-1] if frames else Image.new("RGB", (IMG_SIZE, IMG_SIZE)))
        if len(frames) > self.n_frames:
            idx    = np.linspace(0, len(frames)-1, self.n_frames, dtype=int)
            frames = [frames[i] for i in idx]
        return frames[:self.n_frames]

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        vid_id, label = self.samples[idx]
        clip = self.transform(self._load_frames(vid_id))
        return {"clip": clip, "label": torch.tensor(label, dtype=torch.long), "video_id": vid_id}


def mixup_batch(clips, labels, num_classes, alpha=0.3):
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(clips.size(0), device=clips.device)
    mixed = lam * clips + (1 - lam) * clips[idx]
    oh    = torch.zeros(clips.size(0), num_classes, device=clips.device)
    oh.scatter_(1, labels.unsqueeze(1), 1)
    return mixed, lam * oh + (1 - lam) * oh[idx]


def make_sampler(labels):
    counts  = Counter(labels)
    weights = [1.0 / counts[l] for l in labels]
    return WeightedRandomSampler(weights, len(weights), replacement=True)


def compute_class_weights(labels, num_classes=5):
    counts  = Counter(labels)
    total   = len(labels)
    w = torch.tensor([total / (num_classes * counts.get(i, 1)) for i in range(num_classes)],
                     dtype=torch.float32)
    return (w / w.sum() * num_classes).to(DEVICE)


def oversample_df(df, strategy="median"):
    counts   = df["Target"].value_counts()
    target_n = int(counts.median()) if strategy == "median" else int(counts.max())
    parts = [df]
    for cls, cnt in counts.items():
        if cnt < target_n:
            extra = df[df["Target"] == cls].sample(n=target_n - cnt, replace=True, random_state=42)
            parts.append(extra)
    return pd.concat(parts, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print("Dataset y transforms definidos ✅")

## 4 · Preprocesamiento de videos

In [ ]:
from preprocess import load_yolo, process_video
yolo = load_yolo("yolov8m.pt")
print("✅ YOLO listo")

In [ ]:
def preprocess_split(video_dir, split, yolo_model):
    out_root    = PROCESSED_DIR / split
    video_files = sorted(Path(video_dir).glob(f"*{CFG['video_ext']}"))
    if not video_files:
        video_files = sorted(Path(video_dir).glob("*.mp4"))
    print(f"[{split.upper()}] {len(video_files)} videos")
    for vf in tqdm(video_files, desc=split):
        out = out_root / vf.stem
        if out.exists() and len(list(out.glob("*.png"))) == CFG["n_frames"]:
            continue
        process_video(str(vf), yolo_model, str(out), vf.stem, CFG["n_frames"])
    print("  ✅ Listo")

preprocess_split(TRAIN_VIDEO_DIR, "train", yolo)
preprocess_split(TEST_VIDEO_DIR,  "test",  yolo)

del yolo; torch.cuda.empty_cache()
print("🧹 YOLO liberado")

## 5 · Utilidades de entrenamiento

In [ ]:
def cosine_warmup(optimizer, warmup, total):
    def f(ep):
        if ep < warmup: return (ep + 1) / warmup
        p = (ep - warmup) / max(1, total - warmup)
        return 0.5 * (1 + math.cos(math.pi * p))
    return LambdaLR(optimizer, f)

class EarlyStopping:
    def __init__(self, patience=7, delta=1e-4):
        self.p = patience; self.d = delta
        self.c = 0; self.best = None
    def step(self, s):
        if self.best is None or s > self.best + self.d:
            self.best = s; self.c = 0; return False
        self.c += 1; return self.c >= self.p

def make_optimizer(model, lr):
    decay, nodecay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (nodecay if ("bias" in n or "norm" in n or "bn" in n) else decay).append(p)
    return AdamW([{"params": decay, "weight_decay": CFG["weight_decay"]},
                  {"params": nodecay, "weight_decay": 0.0}], lr=lr)


def run_epoch(model, loader, optimizer, criterion, scaler, is_train):
    model.train() if is_train else model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    ctx = torch.enable_grad() if is_train else torch.no_grad()

    with ctx:
        for batch in tqdm(loader, desc=" Train" if is_train else "  Val ", leave=False):
            clips  = batch["clip"].to(DEVICE, non_blocking=True)
            labels = batch["label"].to(DEVICE, non_blocking=True)

            use_mix = is_train and (np.random.random() < CFG["mixup_prob"])
            if use_mix:
                clips, labels_s = mixup_batch(clips, labels, CFG["num_classes"], CFG["mixup_alpha"])

            if is_train: optimizer.zero_grad(set_to_none=True)

            with autocast("cuda"):
                logits = model(clips)
                loss   = criterion(logits, labels_s if use_mix else labels)

            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG["max_grad_norm"])
                scaler.step(optimizer); scaler.update()

            total_loss += loss.item()
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(batch["label"].numpy())

    f1_pc = f1_score(all_labels, all_preds, average=None, zero_division=0)
    return {"loss": total_loss / len(loader), "macro_f1": float(f1_pc.mean()), "f1_pc": f1_pc.tolist()}


def train_fold(train_df, val_df, fold_n, pseudo_df=None):
    """
    Entrena un fold en 2 fases:
    Fase 1: backbone congelado, solo clasificador
    Fase 2: fine-tuning de los últimos N bloques
    Retorna el mejor F1 de validación y el path del checkpoint.
    """
    print(f"\n{'─'*60}")
    print(f"  FOLD {fold_n}  |  Train: {len(train_df)}  |  Val: {len(val_df)}")
    if pseudo_df is not None:
        print(f"  + {len(pseudo_df)} pseudo-labels")
    print(f"{'─'*60}")

    train_os = oversample_df(train_df)
    if pseudo_df is not None:
        # Los pseudo-labels se añaden sin oversample extra
        train_ds = SheepDataset(str(PROCESSED_DIR / "train"), train_os,
                                pseudo_df=pseudo_df, n_frames=CFG["n_frames"], is_train=True)
    else:
        train_ds = SheepDataset(str(PROCESSED_DIR / "train"), train_os,
                                n_frames=CFG["n_frames"], is_train=True)
    val_ds   = SheepDataset(str(PROCESSED_DIR / "train"), val_df,
                            n_frames=CFG["n_frames"], is_train=False)

    sampler      = make_sampler(train_os["Target"].tolist())
    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], sampler=sampler,
                              num_workers=CFG["num_workers"], pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False,
                              num_workers=CFG["num_workers"], pin_memory=True)

    best_f1   = 0.0
    ckpt_path = CHECKPOINT_DIR / f"fold{fold_n}.pt"
    history   = []

    for phase, (n_ep, lr, unfreeze) in enumerate([
        (CFG["epochs_phase1"], CFG["lr_phase1"], 0),
        (CFG["epochs_phase2"], CFG["lr_phase2"], CFG["unfreeze_phase2"]),
    ], 1):
        print(f"\n  ── Fase {phase}: lr={lr}, unfreeze={unfreeze} bloques, {n_ep} épocas ──")
        model = build_model(unfreeze).to(DEVICE)

        # En fase 2, cargar pesos de fase 1
        if phase == 2 and ckpt_path.exists():
            state = torch.load(ckpt_path, map_location=DEVICE)["model_state"]
            state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}
            model.load_state_dict(state, strict=False)
            print("  Pesos de fase 1 cargados")

        if torch.__version__ >= "2.0.0":
            try: model = torch.compile(model, mode="reduce-overhead")
            except: pass

        cw        = compute_class_weights(train_os["Target"].tolist())
        criterion = SmoothedCE(CFG["label_smoothing"], CFG["num_classes"])
        optimizer = make_optimizer(model, lr)
        scheduler = cosine_warmup(optimizer, CFG["warmup_epochs"], n_ep)
        scaler    = GradScaler("cuda")
        es        = EarlyStopping(CFG["patience"])

        for ep in range(1, n_ep + 1):
            lr_now = optimizer.param_groups[0]["lr"]
            tr = run_epoch(model, train_loader, optimizer, criterion, scaler, True)
            va = run_epoch(model, val_loader,   optimizer, criterion, scaler, False)
            scheduler.step()

            history.append({"fold": fold_n, "phase": phase, "epoch": ep,
                            "train_f1": tr["macro_f1"], "val_f1": va["macro_f1"],
                            "train_loss": tr["loss"],   "val_loss": va["loss"]})

            tag = " ✓ BEST" if va["macro_f1"] > best_f1 else ""
            print(f"  Ph{phase} Ep{ep:3d} | lr={lr_now:.1e} | "
                  f"Tr L={tr['loss']:.3f} F1={tr['macro_f1']:.3f} | "
                  f"Va L={va['loss']:.3f} F1={va['macro_f1']:.3f}{tag}")
            print(f"    F1/clase: {[f'{v:.2f}' for v in va['f1_pc']]}")

            if va["macro_f1"] > best_f1:
                best_f1 = va["macro_f1"]
                torch.save({"model_state": model.state_dict(),
                            "val_f1": best_f1, "cfg": CFG}, ckpt_path)
            if es.step(va["macro_f1"]):
                print(f"  ⏹ Early stop (fase {phase}, época {ep})")
                break

        del model; torch.cuda.empty_cache()

    return best_f1, str(ckpt_path), history

print("Utilidades definidas ✅")

## 6 · K-Fold — Ronda 1 (solo con train.csv)

In [ ]:
df = pd.read_csv(LABEL_CSV)
df.columns = ["Id", "Target"]
print(f"Train: {len(df)} videos")
print(df["Target"].value_counts().sort_index())

skf = StratifiedKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])

fold_results   = []
fold_ckpts     = []
all_history    = []

print(f"\n{'='*60}")
print(f"  RONDA 1: K-Fold con datos de train únicamente")
print(f"{'='*60}")

for fold, (tr_idx, va_idx) in enumerate(skf.split(df["Id"], df["Target"])):
    set_seed(CFG["seed"] + fold)
    tr_df = df.iloc[tr_idx].reset_index(drop=True)
    va_df = df.iloc[va_idx].reset_index(drop=True)
    best_f1, ckpt, history = train_fold(tr_df, va_df, fold + 1)
    fold_results.append(best_f1)
    fold_ckpts.append(ckpt)
    all_history.extend(history)
    print(f"  ✅ Fold {fold+1} → Best Val F1 = {best_f1:.4f}")

print(f"\n{'='*60}")
print(f"  K-Fold Ronda 1 completo")
for i, f1 in enumerate(fold_results): print(f"  Fold {i+1}: {f1:.4f}")
print(f"  Media: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")
print(f"{'='*60}")

## 7 · Pseudo-labeling del test
Usamos el ensemble de los 5 folds para predecir el test y nos quedamos solo con las predicciones de alta confianza (prob >= 0.85). Esas se usan como etiquetas adicionales en la Ronda 2.

In [ ]:
def load_model_for_inference(ckpt_path):
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    cfg_  = ckpt.get("cfg", CFG)
    m = SheepClassifier(
        num_classes=cfg_["num_classes"], n_frames=cfg_["n_frames"],
        backbone_name=cfg_["backbone"],  dropout=0.0, unfreeze_last_n=0,
    ).to(DEVICE)
    state = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model_state"].items()}
    m.load_state_dict(state, strict=False)
    m.eval()
    return m


@torch.no_grad()
def ensemble_predict_proba(models, frames_pil, n_tta=4):
    """Ensemble + TTA → array de probabilidades (num_classes,)"""
    all_p = []
    tf_v  = TemporalConsistentTransform(is_train=False)
    tf_a  = TemporalConsistentTransform(is_train=True)
    for m in models:
        clip = tf_v(frames_pil).unsqueeze(0).to(DEVICE)
        with autocast("cuda"): all_p.append(F.softmax(m(clip), dim=-1).cpu().numpy())
        for _ in range(n_tta - 1):
            clip = tf_a(frames_pil).unsqueeze(0).to(DEVICE)
            with autocast("cuda"): all_p.append(F.softmax(m(clip), dim=-1).cpu().numpy())
    return np.mean(all_p, axis=0)[0]  # (num_classes,)


# Cargar modelos de ronda 1
torch.backends.cudnn.benchmark = True
r1_models = [load_model_for_inference(p) for p in fold_ckpts]
print(f"\n{len(r1_models)} modelos de Ronda 1 cargados")

# Predecir test
test_ds = SheepDataset(str(PROCESSED_DIR / "test"), n_frames=CFG["n_frames"], is_train=False)
print(f"Videos de test: {len(test_ds)}")

pseudo_records = []
all_test_probs = []

for idx in tqdm(range(len(test_ds)), desc="Pseudo-labeling"):
    vid_id = test_ds.samples[idx][0]
    frames = test_ds._load_frames(vid_id)
    probs  = ensemble_predict_proba(r1_models, frames, n_tta=4)
    conf   = float(probs.max())
    pred   = int(probs.argmax())
    all_test_probs.append({"Id": vid_id, "Predicted": pred, "Confidence": conf})
    if conf >= CFG["pseudo_confidence"]:
        pseudo_records.append({"Id": vid_id, "Target": pred})

pseudo_df  = pd.DataFrame(pseudo_records)
all_preds_df = pd.DataFrame(all_test_probs)

print(f"\n✅ Pseudo-labels generados")
print(f"  Videos de test      : {len(all_test_probs)}")
print(f"  Con confianza ≥ {CFG['pseudo_confidence']}: {len(pseudo_df)} ({100*len(pseudo_df)/len(all_test_probs):.0f}%)")
print(f"  Distribución pseudo-labels:")
if len(pseudo_df) > 0:
    print(pseudo_df["Target"].value_counts().sort_index())

# Guardar para análisis
all_preds_df.to_csv(CHECKPOINT_DIR / "r1_test_predictions.csv", index=False)

# Liberar modelos de r1
del r1_models; torch.cuda.empty_cache()

## 8 · K-Fold — Ronda 2 (train + pseudo-labels)

In [ ]:
# Si no hay suficientes pseudo-labels, ajustar confianza automáticamente
if len(pseudo_df) < 20:
    lower_conf = CFG["pseudo_confidence"] - 0.1
    print(f"⚠️  Pocos pseudo-labels. Bajando confianza a {lower_conf}")
    pseudo_df = pd.DataFrame([
        {"Id": r["Id"], "Target": r["Predicted"]}
        for _, r in all_preds_df.iterrows() if r["Confidence"] >= lower_conf
    ])
    print(f"  Pseudo-labels con conf ≥ {lower_conf}: {len(pseudo_df)}")

print(f"\n{'='*60}")
print(f"  RONDA 2: K-Fold con train + {len(pseudo_df)} pseudo-labels")
print(f"{'='*60}")

# Los pseudo-labels usan la carpeta de test procesado
# Necesitamos un dataset combinado
# Truco: creamos un symlink o apuntamos al directorio correcto en SheepDataset
# La solución más limpia: usar processed_dir="train" para labels reales
# y processed_dir="test" para pseudo → manejar en el dataset

# Adaptamos pseudo_df para indicar que sus frames están en /test
# Añadimos un prefijo especial que SheepDataset resolverá
pseudo_df_r2 = pseudo_df.copy()
# Creamos symlinks de test → train para que el dataset los encuentre
import shutil
pseudo_count = 0
for _, row in pseudo_df_r2.iterrows():
    src = PROCESSED_DIR / "test"  / str(row["Id"])
    dst = PROCESSED_DIR / "train" / str(row["Id"])
    if src.exists() and not dst.exists():
        shutil.copytree(str(src), str(dst))
        pseudo_count += 1
print(f"  {pseudo_count} carpetas de pseudo-labels copiadas a processed/train")

r2_results  = []
r2_ckpts    = []
r2_history  = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(df["Id"], df["Target"])):
    set_seed(CFG["seed"] + fold + 100)
    tr_df = df.iloc[tr_idx].reset_index(drop=True)
    va_df = df.iloc[va_idx].reset_index(drop=True)
    # Pasar pseudo_df solo a train, nunca a val
    best_f1, ckpt, history = train_fold(tr_df, va_df, fold + 1, pseudo_df=pseudo_df_r2)
    # Renombrar checkpoint para no pisar ronda 1
    new_ckpt = str(CHECKPOINT_DIR / f"r2_fold{fold+1}.pt")
    import os; os.rename(ckpt, new_ckpt)
    r2_results.append(best_f1)
    r2_ckpts.append(new_ckpt)
    r2_history.extend(history)
    print(f"  ✅ R2 Fold {fold+1} → Best Val F1 = {best_f1:.4f}")

print(f"\n{'='*60}")
print(f"  K-Fold Ronda 2 completo")
for i, f1 in enumerate(r2_results): print(f"  Fold {i+1}: {f1:.4f}")
print(f"  Media R1: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")
print(f"  Media R2: {np.mean(r2_results):.4f} ± {np.std(r2_results):.4f}")
print(f"{'='*60}")

## 9 · Curvas de entrenamiento

In [ ]:
hist_r1 = pd.DataFrame(all_history)
hist_r2 = pd.DataFrame(r2_history)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = ["steelblue", "tomato", "seagreen", "darkorange", "purple"]

for rnd, (hist, results, label) in enumerate([
    (hist_r1, fold_results, "R1"),
    (hist_r2, r2_results,   "R2"),
], 1):
    for fold_n in range(1, CFG["n_folds"] + 1):
        fd = hist[hist["fold"] == fold_n]
        ls = "-" if rnd == 2 else "--"
        c  = colors[fold_n - 1]
        axes[0].plot(range(len(fd)), fd["val_loss"],  color=c, ls=ls, alpha=0.7)
        axes[1].plot(range(len(fd)), fd["val_f1"],    color=c, ls=ls,
                     label=f"F{fold_n} {label}")

axes[0].set_title("Val Loss (-- R1, — R2)")
axes[1].set_title("Val Macro F1 (-- R1, — R2)")
axes[1].axhline(np.mean(r2_results), color="black", ls=":",
                label=f"R2 media={np.mean(r2_results):.3f}")
for ax in axes: ax.set_xlabel("Época"); ax.grid(True, alpha=0.3)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / "curves_r1_r2.png", dpi=120)
plt.show()

## 10 · Inferencia final: ensemble R1 + R2 con TTA

In [ ]:
# Elegir qué checkpoints usar: R2 si mejoró, sino R1
use_r2 = np.mean(r2_results) >= np.mean(fold_results) - 0.01
best_ckpts = r2_ckpts if use_r2 else fold_ckpts
print(f"Usando checkpoints de {'Ronda 2' if use_r2 else 'Ronda 1'}")
print(f"  R1 media: {np.mean(fold_results):.4f}")
print(f"  R2 media: {np.mean(r2_results):.4f}")

# Cargar ensemble
torch.backends.cudnn.benchmark = True
final_models = [load_model_for_inference(p) for p in best_ckpts]
print(f"\n{len(final_models)} modelos cargados para inferencia")

# Re-cargar test_ds (puede haber cambiado el directorio)
test_ds = SheepDataset(str(PROCESSED_DIR / "test"), n_frames=CFG["n_frames"], is_train=False)

results = []
for idx in tqdm(range(len(test_ds)), desc="Inferencia final TTA"):
    vid_id = test_ds.samples[idx][0]
    frames = test_ds._load_frames(vid_id)
    probs  = ensemble_predict_proba(final_models, frames, n_tta=CFG["tta_augments"])
    results.append({"Id": vid_id, "Predicted": int(probs.argmax())})

submission = pd.DataFrame(results)
print(f"\n✅ {len(submission)} predicciones")
print("Distribución:", dict(submission["Predicted"].value_counts().sort_index()))

In [ ]:
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

df_check = pd.read_csv(SUBMISSION_PATH)
assert list(df_check.columns) == ["Id", "Predicted"]
assert df_check["Predicted"].between(0, 4).all()
assert df_check["Id"].nunique() == len(df_check)
assert df_check.isnull().sum().sum() == 0

print(f"✅ Submission válido · {len(df_check)} filas")
print(f"💾 Guardado: {SUBMISSION_PATH}")
submission.head(10)